# Parking Demand Prediction Model

This notebook builds a classification model that predicts the **demand level** (Low / Medium / High) of parking activity for a given zone at a specific hour and day of the week, using the cleaned dataset produced in `01_data_cleaning.ipynb` and the patterns identified in `02_eda_expirement.ipynb`.

**Scope note.** The input `demand_df` is already aggregated by zone, day-of-week, hour, and weekend flag. This means the model learns the *average weekly demand profile* per zone rather than forecasting future dates. This is appropriate as a first classification baseline and supports the MOP use case of informing smart signage with expected demand levels, but it does not constitute a time-series forecast. A short discussion of how to extend this to event-level forecasting is included at the end.

**Target.** `demand_level` with three classes: Low (< 0.33), Medium (0.33–0.66), High (>= 0.66). The EDA flagged class imbalance, so macro-F1 is used as the primary metric rather than accuracy.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    ConfusionMatrixDisplay,
)

import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 1. Load the Cleaned Dataset

The demand dataset produced at the end of `01_data_cleaning.ipynb` is loaded. Each row represents the average occupancy for a unique combination of zone, day-of-week, hour, and weekend flag.

In [ ]:
demand_df = pd.read_csv("../data/processed/cleaned_demand_data.csv")

print("Shape:", demand_df.shape)
demand_df.head()

In [ ]:
demand_df.info()

In [ ]:
print("Class distribution (target):")
print(demand_df["demand_level"].value_counts())
print("\nClass proportions:")
print(demand_df["demand_level"].value_counts(normalize=True).round(3))

### Interpretation

The class counts confirm the imbalance surfaced in the EDA. This directly affects model choice and evaluation: a model that always predicts the majority class can still achieve moderate accuracy, so macro-averaged F1 (which weights classes equally) is a fairer measure of performance.

## 2. Feature Engineering

Four raw features are available: `zone_number`, `status_day`, `status_hour`, `is_weekend`. These are transformed so that a model can use them effectively:

- `status_hour` is cyclic (hour 23 is adjacent to hour 0). Encoding it as a single integer would make the model treat midnight and 11 PM as maximally far apart. A sine/cosine encoding preserves the cyclic relationship.
- `status_day` is ordinal (Monday=0 … Sunday=6) and also cyclic, so it is encoded the same way.
- `zone_number` is high-cardinality categorical. One-hot encoding would explode the feature space. Frequency encoding (replacing each zone with how often it appears) is used as a compact alternative.
- `is_weekend` is already boolean and is cast to integer.

In [ ]:
model_df = demand_df.copy()

# Map day name to number (0 = Monday, 6 = Sunday)
day_to_int = {
    "Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3,
    "Friday": 4, "Saturday": 5, "Sunday": 6,
}
model_df["day_int"] = model_df["status_day"].map(day_to_int)

# Cyclic encoding for hour and day
model_df["hour_sin"] = np.sin(2 * np.pi * model_df["status_hour"] / 24)
model_df["hour_cos"] = np.cos(2 * np.pi * model_df["status_hour"] / 24)
model_df["day_sin"] = np.sin(2 * np.pi * model_df["day_int"] / 7)
model_df["day_cos"] = np.cos(2 * np.pi * model_df["day_int"] / 7)

# Frequency encoding for zone_number
zone_freq = model_df["zone_number"].value_counts(normalize=True).to_dict()
model_df["zone_freq"] = model_df["zone_number"].map(zone_freq)

# Boolean to int
model_df["is_weekend_int"] = model_df["is_weekend"].astype(int)

model_df.head()

In [ ]:
feature_cols = [
    "zone_number",
    "zone_freq",
    "status_hour",
    "hour_sin",
    "hour_cos",
    "day_int",
    "day_sin",
    "day_cos",
    "is_weekend_int",
]

X = model_df[feature_cols].copy()
y_raw = model_df["demand_level"].copy()

# Encode target with a fixed order so Low < Medium < High is preserved in readouts
target_order = ["Low", "Medium", "High"]
label_encoder = LabelEncoder()
label_encoder.fit(target_order)
y = label_encoder.transform(y_raw)

print("Feature matrix shape:", X.shape)
print("Target classes (encoded order):", list(label_encoder.classes_))

## 3. Train / Test Split

A stratified 80/20 split keeps the class balance of Low/Medium/High roughly the same in both partitions. Stratification is important here because of the class imbalance; an unstratified split could accidentally over- or under-represent the minority class in the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train size:", X_train.shape[0])
print("Test size: ", X_test.shape[0])

print("\nTrain class distribution:")
print(pd.Series(label_encoder.inverse_transform(y_train)).value_counts(normalize=True).round(3))
print("\nTest class distribution:")
print(pd.Series(label_encoder.inverse_transform(y_test)).value_counts(normalize=True).round(3))

## 4. Naive Baseline

Before training any real model, a trivial baseline is established: always predict the most common class. Any useful model must beat this.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("Baseline (majority class) performance:")
print(f"  Accuracy : {accuracy_score(y_test, baseline_pred):.4f}")
print(f"  Macro F1 : {f1_score(y_test, baseline_pred, average='macro'):.4f}")

## 5. Candidate Models

Four classifiers are trained and compared:

1. **Logistic Regression** — simple linear baseline. `class_weight='balanced'` down-weights majority-class errors.
2. **Decision Tree** — captures non-linear splits; useful for interpretability.
3. **Random Forest** — ensemble of trees; generally robust on tabular data with mixed feature types.
4. **Gradient Boosting** — sequential boosting; often the strongest tabular classifier at this scale.

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "DecisionTree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=200,
        max_depth=3,
        random_state=RANDOM_STATE,
    ),
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    results.append({"model": name, "accuracy": acc, "macro_f1": macro_f1})

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False).reset_index(drop=True)
results_df

### Interpretation

The ranking by macro-F1 shows which model handles the minority class best, not just which one is most accurate overall. Tree-based ensembles are expected to outperform the linear baseline because the interaction between hour, day and zone is non-linear (e.g. a CBD zone behaves differently at 9 AM on a weekday than a residential zone at the same time).

## 6. Cross-Validation

A single train/test split can give an optimistic or pessimistic read depending on the split. Stratified 5-fold cross-validation on the best model provides a more stable estimate of performance and shows its variance across folds.

In [ ]:
best_name = results_df.iloc[0]["model"]
print(f"Best model on the held-out set: {best_name}")

# Re-create a fresh instance of the best model
best_model = models[best_name]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(best_model, X, y, cv=cv, scoring="f1_macro", n_jobs=-1)

print(f"\nStratified 5-fold macro-F1 scores: {np.round(cv_scores, 4)}")
print(f"Mean : {cv_scores.mean():.4f}")
print(f"Std  : {cv_scores.std():.4f}")

## 7. Detailed Evaluation of the Best Model

The classification report shows per-class precision, recall and F1 so that weak classes (typically the minority) can be identified. The confusion matrix shows which classes are confused for which — for parking management, confusing Medium with High is less costly than confusing High with Low, since the former still triggers sensible signage behaviour.

In [ ]:
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("Classification report (test set):")
print(classification_report(
    y_test, y_pred,
    target_names=label_encoder.classes_,
    digits=3,
))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

## 8. Feature Importance

For tree-based models, feature importance quantifies how much each feature contributed to reducing impurity across splits. This is useful both as a sanity check (the EDA said hour and zone matter — do the importances agree?) and as input to future feature engineering.

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(
        best_model.feature_importances_, index=feature_cols
    ).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(8, 5))
    importances.plot(kind="barh", ax=ax)
    ax.set_title(f"Feature Importances — {best_name}")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.show()
    print(importances.sort_values(ascending=False))
else:
    # Fall back to logistic regression coefficients if the best model is linear
    coefs = pd.DataFrame(
        best_model.coef_, columns=feature_cols, index=label_encoder.classes_
    )
    print("Coefficients per class:")
    print(coefs.T)

### Interpretation

The importance ranking should broadly match the EDA: hour of day and zone (captured through `zone_number` and `zone_freq`) should dominate, with `is_weekend` and the cyclic hour encodings adding incremental signal. If a feature that was clearly predictive in the EDA ranks low here, it suggests either redundancy with another feature (for example, `hour_sin`/`hour_cos` overlapping with `status_hour`) or that the model is not making good use of it.

## 9. Persist the Model and Encoder

The trained model, the target encoder, and the zone-frequency mapping are saved so that the same preprocessing can be applied at inference time — for example, inside the MOP dashboard or a smart-signage service.

In [ ]:
os.makedirs("../models", exist_ok=True)

artifacts = {
    "model": best_model,
    "label_encoder": label_encoder,
    "feature_cols": feature_cols,
    "zone_freq_map": zone_freq,
    "day_to_int": day_to_int,
    "target_order": target_order,
}

joblib.dump(artifacts, "../models/parking_demand_classifier.joblib")
print("Saved: ../models/parking_demand_classifier.joblib")

## 10. Inference Helper

A small helper shows how the saved artifacts are reused to classify an arbitrary (zone, day, hour) query. This is the contract that a downstream MOP service would rely on.

In [ ]:
def predict_demand(zone_number, day_name, hour, artifacts):
    day_int = artifacts["day_to_int"][day_name]
    is_weekend = 1 if day_int >= 5 else 0
    zone_freq_val = artifacts["zone_freq_map"].get(zone_number, 0.0)

    row = {
        "zone_number": zone_number,
        "zone_freq": zone_freq_val,
        "status_hour": hour,
        "hour_sin": np.sin(2 * np.pi * hour / 24),
        "hour_cos": np.cos(2 * np.pi * hour / 24),
        "day_int": day_int,
        "day_sin": np.sin(2 * np.pi * day_int / 7),
        "day_cos": np.cos(2 * np.pi * day_int / 7),
        "is_weekend_int": is_weekend,
    }
    X_row = pd.DataFrame([row])[artifacts["feature_cols"]]
    pred = artifacts["model"].predict(X_row)[0]
    return artifacts["label_encoder"].inverse_transform([pred])[0]


# Example queries — replace with real zone_number values from the data
sample_zone = demand_df["zone_number"].iloc[0]
print("Example predictions:")
for day, hour in [("Monday", 9), ("Monday", 14), ("Saturday", 2), ("Friday", 18)]:
    level = predict_demand(sample_zone, day, hour, artifacts)
    print(f"  Zone {sample_zone}, {day} {hour:02d}:00  ->  {level}")

## 11. Limitations and Extension to Event-Level Forecasting

The classifier above reproduces the aggregated demand profile well, but this is partly a feature of the dataset rather than true predictive skill. The input was already averaged across all dates, so the model is effectively learning a lookup table keyed on (zone, day, hour). Scores on this target should be reported with that caveat.

**To produce a genuinely predictive model, the next iteration should:**

1. Work directly from `merged_bay_sensor_data.csv` (one row per sensor reading) rather than `cleaned_demand_data.csv`.
2. Use a date-based train/test split — for example, train on weeks 1–3 of each month and test on week 4 — so performance reflects generalization to *unseen time*, not unseen rows from the same aggregation.
3. Predict bay-level `occupied` (binary) at a short horizon (e.g. 15 minutes ahead) using lag features: occupancy of the same bay in the previous 1/2/3 intervals, rolling mean of the zone in the last hour, and the day/hour/restriction features already engineered here.
4. Evaluate with PR-AUC or F1 on the positive class (occupied), since availability prediction is what directly supports the smart-signage and app-integration use cases in the project brief.

**Additional improvements worth considering:**

- Add `restriction_display` and `restriction_days` from `merged_bay_df` as features — the EDA showed restriction type clearly affects occupancy.
- Incorporate events/public-holiday flags once that data is available — the MOP brief lists "nearby events" as an input the final system should consider.
- Evaluate whether a regression on `average_occupancy` (a continuous target) is more useful than the 3-class binning, since the Low/Medium/High cut-offs at 0.33 and 0.66 are arbitrary and lose information.